# Q2: Data Cleaning

**Phase 3:** Data Cleaning & Preprocessing  
**Points: 9 points**

**Focus:** Handle missing data, outliers, validate data types, remove duplicates.

**Lecture Reference:** Lecture 11, Notebook 1 ([`11/demo/01_setup_exploration_cleaning.ipynb`](https://github.com/christopherseaman/datasci_217/blob/main/11/demo/01_setup_exploration_cleaning.ipynb)), Phase 3. Also see Lecture 05 (data cleaning).

---

## Setup

In [8]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import datetime

# Load data from Q1 (or directly from source)
df = pd.read_csv('data/beach_sensors.csv')
# If you saved cleaned data from Q1, you can load it:
# df = pd.read_csv('output/q1_exploration.csv')  # This won't work - load original

In [9]:
#Q1.1
# check datatypes in dataset
original_df = df.copy()
df_dtypes = df.dtypes
print(f"These are the datatypes:\n{df_dtypes}")

# change Measurement Timestamp to datetime
df['Measurement Timestamp'] = pd.to_datetime(df['Measurement Timestamp'])
# change precipitation type to categories
df['Precipitation Type'] = df['Precipitation Type'].astype('category')
precipitation_cats = df['Precipitation Type'].cat.categories
print(f'These are the categories for Precipitation Type:\n {precipitation_cats}')
precipitation_map = {
    0:0,
    5:1,
    60:2,
    70:3
}
df['Precipitation Type'] = df['Precipitation Type'].map(precipitation_map)

# handle duplicates
dups = df.duplicated().sum() 
print(f"There are {dups} duplicates")

# check and handle missing values in dataset
missing = df.isna().sum()
print(f"These are all the missing values:\n {missing}")


These are the datatypes:
Station Name                    object
Measurement Timestamp           object
Air Temperature                float64
Wet Bulb Temperature           float64
Humidity                         int64
Rain Intensity                 float64
Interval Rain                  float64
Total Rain                     float64
Precipitation Type             float64
Wind Direction                   int64
Wind Speed                     float64
Maximum Wind Speed             float64
Barometric Pressure            float64
Solar Radiation                  int64
Heading                        float64
Battery Life                   float64
Measurement Timestamp Label     object
Measurement ID                  object
dtype: object
These are the categories for Precipitation Type:
 Index([0.0, 5.0, 60.0, 70.0], dtype='float64')
There are 0 duplicates
These are all the missing values:
 Station Name                       0
Measurement Timestamp              0
Air Temperature               

In [10]:
#Q1.2
# TODO:handle negative values
df['Rain Intensity'] = df['Rain Intensity'].clip(lower=0, upper=5)
df['Interval Rain'] = df['Interval Rain'].clip(lower=0,upper=25) # give reasoning for upper bound
#df['Total Rain'] = df['Total Rain'].clip(lower=0, upper=200)
df['Solar Radiation'] = df['Solar Radiation'].clip(lower=0, upper=None) # give reasoning for upper bound
# TODO:handle extremely high values
df['Wind Speed'] = df['Wind Speed'].clip(lower=0, upper=18)# give reasoning for upper bound (highest value ever recorded was 39.. in 1894)
df['Maximum Wind Speed'] = df['Maximum Wind Speed'].clip(lower=0, upper=24) # give reasoning for upper bound (highest value ever recorded was 39, there were more 'high values than the 'Wind Speed' col, therefore higher max)
df['Barometric Pressure'] = df['Barometric Pressure'].clip(lower=800, upper=1050) # give reasoning for upper bound
df['Solar Radiation'] = df['Solar Radiation'].clip(lower=0, upper=1100) # give reasoning for upper bound


# TODO:look within columns to figure out which need further handling
df['Precipitation Type']
## Total Rain needs to be adjusted..
# TODO:ffill / bfill


0           0
1           0
2           0
3         NaN
4           0
         ... 
196133      2
196134    NaN
196135      2
196136    NaN
196137      2
Name: Precipitation Type, Length: 196138, dtype: category
Categories (4, int64): [0, 1, 2, 3]

In [11]:
#Q1.3
# TODO:ffill / bfill
cols_lots_missing = ['Wet Bulb Temperature', 'Total Rain', 'Heading', 'Rain Intensity', 'Precipitation Type']# look back at which columns
df[cols_lots_missing] = df[cols_lots_missing].ffill()  #forward filled all with many missing columns

#TODO:drop missing rows for columns with < 5% 
df = df.dropna(subset = ['Barometric Pressure', 'Air Temperature']) #dropping barometric pressure and air temperature missing values because < 5% missing
df_after_missing = df.copy()
print("Forward filled missing values in columns with many missing; dropped rows where Barometric Pressure and Air Temperature were missing")



Forward filled missing values in columns with many missing; dropped rows where Barometric Pressure and Air Temperature were missing


In [12]:
#Q1.4
#TODO:Handle outliers

# check for outliers and filter values beyond 3 standard deviations, for 'Wet Bulb Temperature', 'Total Rain', 'Heading'

df_clean = df_after_missing.copy()

# Select all numeric columns
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns

# Exclude the categorical column and other columns that turn to all 0 (i'm looking at you Rain Intensity and Interval Rain)
exclude_cols = ['Precipitation Type', 'Rain Intensity', 'Interval Rain']
numeric_cols = [col for col in numeric_cols if col not in exclude_cols]

# wait to handle Rain Intensity and Interval Rain with log transformations in Q4
# leave Precipitation Type as is

# Apply IQR filtering
for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]




print(f"Rows after removing outliers: {df_clean.shape[0]}")


df_clean.to_csv('output/q2_cleaned_data.csv', index = False)
print("Saved cleaned data to csv!")

Rows after removing outliers: 121076
Saved cleaned data to csv!


In [13]:
# Q2
# aye yi yi
all_missing_cols = list(cols_lots_missing) + ['Barometric Pressure'] + ['Air Temperature']

def missing_summary(df, col, method, result):
    missing_count = df[col].isna().sum()
    missing_perc = (missing_count / len(df))*100
    txt = (
        f'- {col}: {missing_count} missing values ({missing_perc:.3f}%)\n'
        f'  Method: {method}\n'
        f'  Result: {result}\n'
    )
    return txt
def outlier_summary_iqr(df, col, k=3):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - k * IQR
    upper = Q3 + k * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    txt = (
        f"- {col}: Detected {len(outliers)} outliers using IQR method (threshold ±{k}*IQR)\n"
        f"  Method: Removed rows beyond bounds\n"
        f"  Bounds: [{lower:.1f}, {upper:.1f}]\n"
        f"  Result: {len(outliers)} rows removed\n"
    )
    return txt

with open('output/q2_cleaning_report.txt', 'w') as f:
    f.write("===DATA CLEANING REPORT===\n\n")
    f.write(f'Number of rows before cleaning: {original_df.shape[0]}\n\n')
    f.write(f'Missing Data Handling:\n')
    for col in cols_lots_missing:
        f.write(missing_summary(
            original_df, col,
            'Forward-fill method ws used',
            'All missing values were filled'))
    for col in ['Barometric Pressure', 'Air Temperature']:
        f.write(missing_summary(
            original_df, col,
            'Dropped NA rows because there were so few (<5%)',
            'All Missing values were dropped'
        ))
    f.write('\n')
    f.write('Outlier handling:\n')
    for col in numeric_cols:
        f.write(outlier_summary_iqr(df_after_missing, col))
    f.write('\n')
    f.write('Data Type Conversions:\n - Converted Measurement Timestamp to datetime\n\n')
    f.write(f'Duplicates removed: {dups}\n\n')
    f.write(f'Rows after cleaning: {df_clean.shape[0]}')
print('Saved to txt file!')

Saved to txt file!


In [14]:
# Q3
# number of rows after cleaning
with open('output/q2_rows_cleaned.txt', 'w') as f:
    f.write(str(df_clean.shape[0]))

---

## Objective

Clean the dataset by handling missing data, outliers, validating data types, and removing duplicates.

**Time Series Note:** For time series data, forward-fill (`ffill()`) is often appropriate for missing values since sensor readings are continuous. However, you may choose other strategies based on your analysis.

---

## Required Artifacts

You must create exactly these 3 files in the `output/` directory:

### 1. `output/q2_cleaned_data.csv`
**Format:** CSV file
**Content:** Cleaned dataset with same structure as original (same columns)
**Requirements:**
- Same columns as original dataset
- Missing values handled (filled, dropped, or imputed)
- Outliers handled (removed, capped, or transformed)
- Data types validated and converted
- Duplicates removed
- **Sanity check:** Dataset should retain most rows after cleaning (at least 1,000 rows). If you're removing more than 50% of data, reconsider your strategy—imputation is usually preferable to dropping rows for this dataset.
- **No index column** (save with `index=False`)

### 2. `output/q2_cleaning_report.txt`
**Format:** Plain text file
**Content:** Detailed report of cleaning operations
**Required information:**
- Rows before cleaning: [number]
- Missing data handling method: [description]
  - Which columns had missing data
  - Method used (drop, forward-fill, impute, etc.)
  - Number of values handled
- Outlier handling: [description]
  - Detection method (IQR, z-scores, domain knowledge)
  - Which columns had outliers
  - Method used (remove, cap, transform)
  - Number of outliers handled
- Duplicates removed: [number]
- Data type conversions: [list any conversions]
- Rows after cleaning: [number]

**Example format:**
```
DATA CLEANING REPORT
====================

Rows before cleaning: 50000

Missing Data Handling:
- Water Temperature: 2500 missing values (5.0%)
  Method: Forward-fill (time series appropriate)
  Result: All missing values filled
  
- Air Temperature: 1500 missing values (3.0%)
  Method: Forward-fill, then median imputation for remaining
  Result: All missing values filled

Outlier Handling:
- Water Temperature: Detected 500 outliers using IQR method (3×IQR)
  Method: Capped at bounds [Q1 - 3×IQR, Q3 + 3×IQR]
  Bounds: [-5.2, 35.8]
  Result: 500 values capped

Duplicates Removed: 0

Data Type Conversions:
- Measurement Timestamp: Converted to datetime64[ns]

Rows after cleaning: 50000
```

### 3. `output/q2_rows_cleaned.txt`
**Format:** Plain text file
**Content:** Single integer number (total rows after cleaning)
**Requirements:**
- Only the number, no text, no labels
- No whitespace before or after
- Example: `50000`

---

## Requirements Checklist

- [ ] Missing data handling strategy chosen and implemented
- [ ] Outliers detected and handled (IQR method, z-scores, or domain knowledge)
- [ ] Data types validated and converted
- [ ] Duplicates identified and removed
- [ ] Cleaning decisions documented in report
- [ ] All 3 required artifacts saved with exact filenames

---

## Your Approach

1. **Handle missing data** - Choose appropriate strategy (drop, forward-fill, impute) based on data characteristics
2. **Detect and handle outliers** - Use IQR method or z-scores; decide whether to remove, cap, or transform
3. **Validate data types** - Ensure numeric and datetime columns are properly typed
4. **Remove duplicates**
5. **Document and save** - Write detailed cleaning report explaining your decisions

---

## Decision Points

- **Missing data:** Should you drop rows, impute values, or forward-fill? Consider: How much data is missing? Is it random or systematic? For time series, forward-fill is often appropriate.
- **Outliers:** Are they errors or valid extreme values? Use IQR method or z-scores to detect, then decide: remove, cap, or transform. Document your reasoning.
- **Data types:** Are numeric columns actually numeric? Are datetime columns properly formatted? Convert as needed.

---

## Checkpoint

After Q2, you should have:
- [ ] Missing data handled
- [ ] Outliers addressed
- [ ] Data types validated
- [ ] Duplicates removed
- [ ] All 3 artifacts saved: `q2_cleaned_data.csv`, `q2_cleaning_report.txt`, `q2_rows_cleaned.txt`

---

**Next:** Continue to `q3_data_wrangling.md` for Data Wrangling.
